### Meta/System Prompt Engineering

Let's begin by using our very generic template to write out some product requirements for the AI model.

In your small group, discuss what role you want the model to play, what the task is (ELI5), any rules that apply, any examples of good completion of the task and any additional contextual information the model should use. Context can especially consider domain-specific words or concepts that help activate areas of the model you think are relevant in the model's information.

Test out a few different prompts to see if you like it from a "vibes" perspective.

If you want you can try a different model; however try to focus on analyzing how changing your meta prompt changes the output instead of comparing models at this point.

In [2]:
import ollama
from openai import OpenAI

In [3]:
model_name = 'gemma3:1b'

In [3]:
with open('data/templates/meta_prompt_template', 'r') as mpt:
    meta_prompt_template = mpt.read()

In [5]:
print(meta_prompt_template)


{role_description}

{task_description}

{rules}

{examples}

{context}



In [7]:
role = """write here"""

In [8]:
task = """write here"""

In [9]:
rules = """write here"""

In [10]:
examples = """write here"""

In [11]:
context = """write here"""

In [13]:
meta_prompt = meta_prompt_template.format(
    role_description=role,
    task_description=task,
    rules=rules,
    examples=examples,
    context=context)

In [14]:
print(meta_prompt)


write here

write here

write here

write here

write here



In [15]:
test_prompt = "Can you help me with xxxxx?"

In [16]:
conversation = [
    {'role': 'system',
     'content': meta_prompt},
    {'role': 'user',
     'content': test_prompt}
]
    

In [19]:
response = ollama.chat(
    model=model_name,
    messages=conversation
)

In [22]:
print(response.message.content)

Please provide me with the text you want me to help you with! 😊  I need the text you're referring to. 

I need the content of “xxxx” to be able to assist you.


In [45]:
with open('data/meta_prompts/use_case_metaprompt', 'w') as ucp:
    ucp.write(meta_prompt_template.format(
        role_description=role,
        task_description=task,
        rules=rules,
        examples=examples,
        context=context))

### Task Description to Generate Synthetic Data

Since you probably don't have real data yet for your use case, let's try generating some synthetic data. Obviously, this data will only approximate potential users and will never be as good as real data -- but we'll use it to get a baseline.

In [9]:
synthetic_generation_prompt = """

You are a data generation assistant. Your role is to generate realistic data based on the task description and criteria. 

In this task, users are asking about different recipes they can make with simple ingredients they have at home. 

The users vary from age, cooking ability, gender, language (English and Spanish as primary languages) and the type of food they like.
The one commonality they have is that they need to cook a delicious meal in a short period of time (under an hour) without having to go to the store.

You must:

- generate realistic initial queries for these users, varying the type of food/taste and recipes you ask for
- in the intial query specify what ingredients you have and what you are interested in cooking
- keep it short and simple
- use a variety of different ingredients, but stick to common ones
- use different voices, personalities, languages (half english/half spanish)
- Don't always ask in the same way, use different scenarios and sentence structure to keep it interesting!

Here are a few example user requests:

I'm looking for a mediterranean style dinner. I have salad, normal pantry stuff, tomatoes and carrots. What should I cook?
Me encanta comida picante! Tengo salsa, tomatillos, arroz, frijoles... pero no tengo una idea de que cocinar.. 
My son and I love to have eggs for dinner, but I'm out of creative ideas. Can you help me make a new kid-friendly recipe we can cook together?

You should return a list where each of these user requests is a new entry in a Python list. I need about 15 entries.
"""


In [10]:
response = ollama.chat(
    model=model_name,
    messages = [
        {'role': 'system',
         'content': synthetic_generation_prompt,
        }])

In [11]:
print(response.message.content)

Okay, here's a list of 15 user requests, generated as requested, designed to be realistic and varied, aiming for a range of user profiles and food preferences, all while adhering to the constraints you’ve set.  I’ve focused on simple ingredients and quick cooking times.

```python
[
    "I’m looking for a simple pasta dish. I have spaghetti, olive oil, garlic, and parmesan. What’s a quick and tasty recipe?",
    "¡Hola! Tengo huevos, tortillas, cebolla y cilantro. ¿Podrías sugerirme algo que pueda preparar en 30 minutos?",
    "Looking for something comforting. I have chicken, rice, and frozen peas. Can you suggest a warm and easy dinner?",
    "I want a Mexican dish. I have tortillas, salsa verde, cheese, and olives.  I'm not very good at cooking, but I'd love a recipe that's easy.",
    "I’m craving something savory. I have potatoes, onions, and ham.  What quick recipe can I make with these?",
    "I’m trying to make a dinner for my family. I have ground beef, tortillas, and salsa. W

### Improvement and iteration

- What went well? What do you think we could improve?
- How can we test and compare?

### Your Turn
- Now with your task, let's take these learnings and generate some synthetic data for you!
  

### Save your examples

Now let's save this synthetic data and use it to evaluate our prompts.

In [12]:
examples = [
    "I’m looking for a simple pasta dish. I have spaghetti, olive oil, garlic, and parmesan. What’s a quick and tasty recipe?",
    "¡Hola! Tengo huevos, tortillas, cebolla y cilantro. ¿Podrías sugerirme algo que pueda preparar en 30 minutos?",
    "Looking for something comforting. I have chicken, rice, and frozen peas. Can you suggest a warm and easy dinner?",
    "I want a Mexican dish. I have tortillas, salsa verde, cheese, and olives.  I'm not very good at cooking, but I'd love a recipe that's easy.",
    "I’m craving something savory. I have potatoes, onions, and ham.  What quick recipe can I make with these?",
    "I’m trying to make a dinner for my family. I have ground beef, tortillas, and salsa. What's a quick and satisfying meal?",
    "Hey! I have some chicken breast, potatoes, and lemon.  What’s a healthy and simple recipe I could whip up in under an hour?",
    "I want a flavorful vegetarian meal. I have beans, corn, and peppers.  Can you give me a recipe that won’t take too long?",
    "Looking for a quick and easy snack. I have some crackers, cheese, and grapes.  What can I make in 15 minutes?",
    "I’m craving a hearty soup. I have potatoes, celery, and chicken broth.  Can you suggest a recipe that’s relatively easy?",
    "My son and I love tacos! We have ground beef, tortillas, lettuce, and salsa. What’s a kid-friendly recipe we can make?",
    "I have some sausage, potatoes, and spinach.  What a simple dish could I prepare in under an hour?",
    "I need a quick recipe for a main course. I have ground turkey, rice, and a tomato sauce.  It needs to be really tasty.",
    "Hola, I have some chicken, rice, and steamed broccoli. What can I make that’s quick and delicious?",
    "I want something with vegetables and protein.  I have ground beef, carrots, and onions.  Give me a recipe.",
    "I’m feeling like something spicy. I have chicken, soy sauce, and chili peppers.  What quick and spicy meal can I create? "
]

In [17]:
import csv

with open('data/prompts/syntehtic_initial_prompts.csv', 'w', newline='') as csvfile:
    fieldnames = ['synthetic_prompt']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in examples:
        writer.writerow({'synthetic_prompt': row})

In [18]:
!head data/prompts/syntehtic_initial_prompts.csv

synthetic_prompt
"I’m looking for a simple pasta dish. I have spaghetti, olive oil, garlic, and parmesan. What’s a quick and tasty recipe?"
"¡Hola! Tengo huevos, tortillas, cebolla y cilantro. ¿Podrías sugerirme algo que pueda preparar en 30 minutos?"
"Looking for something comforting. I have chicken, rice, and frozen peas. Can you suggest a warm and easy dinner?"
"I want a Mexican dish. I have tortillas, salsa verde, cheese, and olives.  I'm not very good at cooking, but I'd love a recipe that's easy."
"I’m craving something savory. I have potatoes, onions, and ham.  What quick recipe can I make with these?"
"I’m trying to make a dinner for my family. I have ground beef, tortillas, and salsa. What's a quick and satisfying meal?"
"Hey! I have some chicken breast, potatoes, and lemon.  What’s a healthy and simple recipe I could whip up in under an hour?"
"I want a flavorful vegetarian meal. I have beans, corn, and peppers.  Can you give me a recipe that won’t take too long?"
"Looking fo